<!-- # Neural Discrete Representation Learning -->
# 神经离散表征学习

- [论文](https://arxiv.org/abs/1711.00937)
- [代码](https://github.com/google-deepmind/sonnet/blob/v2/sonnet/src/nets/vqvae.py)

## 摘要

无监督学习中有用的表示学习仍然是一个关键挑战。在本文中，我们提出了一种简单但强大的生成模型，用于学习这种离散表示。我们的模型，向量量化变分自编码器（VQ-VAE），在两个关键方面与 VAEs 不同：编码器网络输出的是离散的而不是连续的代码；先验是学习得到的而不是静态的。为了学习离散的潜在表示，我们结合了向量量化的思想（VQ）。使用 VQ 方法使模型能够绕过“后验崩溃”问题——当潜在变量与强大的自回归解码器配对时被忽略——通常在 VAE 框架中观察到的问题。将这些表示与自回归先验配对，模型可以生成高质量的图像、视频和语音，以及进行高质量的说话人转换和无监督音素学习，这进一步证明了学习到的表示的实用性。


<font color="orange">

核心是为了解决Posterior Collapse(后验崩溃)问题
1. 本质: $q_\phi(z|x) \approx p(z)$, 即输入$x$无法为潜变量$z$提供有效信息，编码器失效。
2. 典型现象:
   - KL 散度项快速降至接近 0，ELBO 只剩重构项主导；
   - 潜变量失去多样性，编码器输出趋同（如高斯 VAE 中均值固定、方差趋近 0）；
   - 采样生成的样本缺乏多样性，仅重复少数模式；
   - 重构看似正常，但潜变量无法捕捉数据语义结构，表征无意义。



## 引言

该文章工作的贡献：

<!-- - Introducing the VQ-VAE model, which is simple, uses discrete latents, does not suffer from “posterior collapse” and has no variance issues.
- We show that a discrete latent model (VQ-VAE) perform as well as its continuous model counterparts in log-likelihood.
- When paired with a powerful prior, our samples are coherent and high quality on a wide variety of applications such as speech and video generation.
- We show evidence of learning language through raw speech, without any supervision, and show applications of unsupervised speaker conversion. -->
- 引入 VQ-VAE 模型，该模型简单、使用离散潜在变量、不会出现“后验坍塌”问题，且没有方差问题。
- 我们证明了离散潜在模型（VQ-VAE）在对数似然方面表现与连续模型相当。
- 当与强大的先验知识结合时，我们的样本在各种应用（如语音和视频生成）上表现出一致性和高质量。
- 我们展示了通过原始语音学习语言的无监督证据，并展示了无监督说话人转换的应用。

## 3 VQ-VAE模型

### 3.1 离散潜变量

我们定义一个潜嵌入空间$e\in \mathbb{R}^{K \times D}$，其中$K$是离散潜空间的大小（即一个 $K$-路分类），$D$是每个潜嵌入向量$e_i$的维度。请注意，存在$K$个嵌入向量$e_i \in \mathbb{R}^D, i\in 1, 2, ...K$。如图 1 所示，模型接收输入$x$，该输入通过编码器产生输出$z_e(x)$。然后通过使用共享嵌入空间$e$进行最近邻查找来计算离散潜变量$z$，如图 1 所示。解码器的输入是对应的嵌入向量$e_k$，如方程 2 所示。可以将这种前向计算流程视为一个常规自编码器，它具有一种特殊的非线性映射，将潜变量映射到 1-of-K 嵌入向量。该模型的完整参数集是编码器、解码器和嵌入空间参数的并集。为了简化，在本节中我们使用单个随机变量$z$来表示离散潜变量，但对于语音、图像和视频，我们分别提取 1D、2D 和 3D 潜特征空间。

<div style="background-color: white; width: 80%; margin: auto;">
    <img src="./assets/Figure1_9.png" />
    <p style="color: black">图 1：左：描述 VQ-VAE 的图示。右：嵌入空间的可视化。编码器的输出被映射到最近的点。梯度（红色）将推动编码器改变其输出，这可能改变下一次前向传递中的配置。</p>
</div>

后验分类分布概率定义为如下的独热编码：

$$\begin{aligned}
q(z=k|x) &= \begin{cases} 1, & \text{for } k = \arg\min_j ||z_e(x) - e_j||_2 \\ 0, & \text{otherwise} \end{cases} &{(1)} \\
z_q(x) &= e_k, \quad \text{where } k = \arg\min_j ||z_e(x) - e_j||_2 &{(2)}
\end{aligned}$$

其中$z_e(x)$是编码器的输出。我们将该模型视作一种变分自编码器（VAE），在这个模型中，我们可以通过证据下界（ELBO）来对对数边缘似然$\log p(x)$进行下界估计（约束）。我们所采用的提议分布（proposal distribution）$q(z=k|x)$是确定性的（非随机的）。并且，通过为隐变量$z$定义一个简单的均匀先验分布（uniform prior），我们得到的 KL 散度（KL divergence）是一个常数，其值等于$\log K$。

### 3.2 学习